In [1]:
from config import *
from data.crossner_utils import read_bio, prepare_tokenized_dataset, label_map
from training.train_utils import load_nuner_model, load_tokenizer_nuner, evaluate_finetune_nuner_model
import logging

logging.getLogger("transformers").setLevel(logging.ERROR)

tokenizer = load_tokenizer_nuner()

for domain in ['ai', 'literature', 'music', 'politics', 'science']:
    
    train_tokens, train_labels = read_bio(f"{CROSSNER_PATH}/{domain}/train.txt")
    dev_tokens, dev_labels = read_bio(f"{CROSSNER_PATH}/{domain}/dev.txt")
    test_tokens, test_labels = read_bio(f"{CROSSNER_PATH}/{domain}/test.txt")

    label2id, id2label = label_map(train_labels + dev_labels + test_labels)

    model = load_nuner_model(
        id2label=id2label,
        label2id=label2id
    )

    train_tok = prepare_tokenized_dataset(train_tokens, train_labels, tokenizer, label2id)
    dev_tok = prepare_tokenized_dataset(dev_tokens, dev_labels, tokenizer, label2id)
    test_tok = prepare_tokenized_dataset(test_tokens, test_labels, tokenizer, label2id)
    
    print(f"Evaluating on \"{domain.upper()}\" domain:")

    evaluate_finetune_nuner_model(
        model=model,
        domain=DOMAIN,
        id2label=id2label,
        label2id=label2id,
        test_tok=test_tok,
        train_tok=train_tok,
        dev_tok=dev_tok,
        tokenizer=tokenizer
    )


Evaluating on "AI" domain:
P: 1.251 R: 5.140 F1: 2.012 Base Model
P: 22.22 R: 25.92 F1: 23.93 Tunned Model
Evaluating on "LITERATURE" domain:
P: 0.437 R: 1.588 F1: 0.685 Base Model
P: 55.03 R: 52.78 F1: 53.88 Tunned Model
Evaluating on "MUSIC" domain:
P: 0.465 R: 1.378 F1: 0.695 Base Model
P: 40.86 R: 48.53 F1: 44.36 Tunned Model
Evaluating on "POLITICS" domain:
P: 0.338 R: 0.974 F1: 0.502 Base Model
P: 64.52 R: 69.58 F1: 66.96 Tunned Model
Evaluating on "SCIENCE" domain:
P: 0.914 R: 2.978 F1: 1.398 Base Model
P: 50.32 R: 58.10 F1: 53.93 Tunned Model


Fine-tune

In [24]:
args = TrainingArguments(
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=40,
    learning_rate=3e-5,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    logging_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=dev_tok,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=5,   # epochs sin mejorar antes de parar
            early_stopping_threshold=0.0
        )
    ]
)

/tmp/ipykernel_2754/1633331591.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [25]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,1.886300,1.199137,0.000000
2,1.100600,1.034347,0.207927
3,0.894800,0.855324,0.274567
4,0.679300,0.723305,0.453390
5,0.502900,0.640747,0.547884
6,0.399300,0.599981,0.540472
7,0.283200,0.546576,0.578110
8,0.211700,0.528376,0.583333
9,0.149300,0.510860,0.602013
10,0.119800,0.507560,0.619771


TrainOutput(global_step=312, training_loss=0.2773440705421261, metrics={'train_runtime': 433.7708, 'train_samples_per_second': 9.221, 'train_steps_per_second': 1.199, 'total_flos': 95139320563056.0, 'train_loss': 0.2773440705421261, 'epoch': 24.0})

In [29]:
trainer.save_model("models/nuner_crossner/final")
tokenizer.save_pretrained("models/nuner_crossner/final")

('models/nuner_crossner/final/tokenizer_config.json',
 'models/nuner_crossner/final/special_tokens_map.json',
 'models/nuner_crossner/final/vocab.json',
 'models/nuner_crossner/final/merges.txt',
 'models/nuner_crossner/final/added_tokens.json',
 'models/nuner_crossner/final/tokenizer.json')

Eval Finetuned Model

In [ ]:
from transformers import TrainingArguments, Trainer, AutoModelForTokenClassification

# 1️⃣ cargar modelo entrenado PRIMERO
model = AutoModelForTokenClassification.from_pretrained(
    "models/nuner_crossner/final",
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id,
)

# 2️⃣ args solo para evaluación
args = TrainingArguments(
    output_dir="tmp_eval",
    per_device_eval_batch_size=8,
    report_to="none",
)

# 3️⃣ trainer con TEST dataset (no dev)
trainer = Trainer(
    model=model,
    args=args,
    eval_dataset=test_tok,
    data_collator=collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# 4️⃣ evaluar
metrics = trainer.evaluate()

print(metrics)

/tmp/ipykernel_2754/3221423009.py:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


{'eval_loss': 0.5417467951774597, 'eval_model_preparation_time': 0.0013, 'eval_precision': 0.6343476018566271, 'eval_recall': 0.6799336650082919, 'eval_f1': 0.656350053361793, 'eval_runtime': 1.1038, 'eval_samples_per_second': 390.455, 'eval_steps_per_second': 48.92}
